# <center> Modelagem </center>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler

from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    roc_auc_score, f1_score,
    precision_score, recall_score,
    roc_curve, confusion_matrix
)

import xgboost as xgb
import warnings
warnings.filterwarnings("ignore")

SEED = 42

In [ ]:
# LOADING DATA
X = pd.read_csv("../data/processed/X_features.csv")
y = pd.read_csv("../data/processed/y_target.csv").squeeze()

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"\nDistribuicao do target:")
print(y.value_counts(normalize=True).round(3))

In [ ]:
# STRATIFIED TRAIN-TEST SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size = 0.2, random_state = SEED, stratify=y
)
print(f"\nTreino: {X_train.shape[0]:,} registros")
print(f"Teste: {X_test.shape[0]:,} registros")

# VERIFYING STRATIFICATION
print(f"\nProporcao de inadimplentes no treino: {y_train.mean():.2%}")
print(f"Proporcao de inadimplentes no teste: {y_test.mean():.2%}")

In [ ]:
# BASELINE MODEL

from sklearn.dummy import DummyClassifier

# PREDICTION OF THE MAJORITARY CLASS 
baseline = DummyClassifier(strategy="most_frequent", random_state=SEED)
baseline.fit(X_train, y_train)
y_pred_baseline = baseline.predict(X_test)
y_proba_baseline = baseline.predict_proba(X_test)[:,1]  

In [ ]:
#RESULTS

print("--- BASELINE - MAJORITY CLASS ---")
print(f"ROC-AUC: {roc_auc_score(y_test, y_proba_baseline):.4f}")
print(f"F1-Score: {f1_score(y_test, y_pred_baseline,zero_division=0):.4f}")
print(f"Recall: {recall_score(y_test, y_pred_baseline,zero_division=0):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_baseline,zero_division=0):.4f}")

In [ ]:
# CREATING A FUNCTION TO EVALUATE MODELS

def calcular_ks(y_true, y_proba):
    """KS — métrica padrão do mercado de crédito brasileiro"""
    from scipy.stats import ks_2samp
    scores_positivos = y_proba[y_true == 1]
    scores_negativos = y_proba[y_true == 0]
    ks_stat, _ = ks_2samp(scores_positivos, scores_negativos)
    return ks_stat

def avaliar_modelo(nome, modelo, X_train, X_test, y_train, y_test, cv=5):
    """
    Avalia um modelo com métricas completas + validação cruzada estratificada.
    Retorna dicionário com todas as métricas.
    """
    # Predições
    y_proba = modelo.predict_proba(X_test)[:, 1]
    y_pred = (y_proba >= 0.5).astype(int)

    # Métricas no teste
    auc = roc_auc_score(y_test, y_proba)
    ks = calcular_ks(y_test, y_proba)
    f1 = f1_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)

    # Validação cruzada estratificada no treino
    skf = StratifiedKFold(n_splits=cv, shuffle=True, random_state=SEED)
    cv_results = cross_validate(
        modelo, X_train, y_train,
        cv=skf,
        scoring="roc_auc",
        return_train_score=True
    )
    cv_auc_mean = cv_results["test_score"].mean()
    cv_auc_std = cv_results["test_score"].std()
    train_auc_mean = cv_results["train_score"].mean()

    print(f"\n{'='*50}")
    print(f"MODELO: {nome}")
    print(f"{'='*50}")
    print(f"ROC-AUC  (teste):       {auc:.4f}")
    print(f"KS       (teste):       {ks:.4f}")
    print(f"F1-Score (teste):       {f1:.4f}")
    print(f"Precision(teste):       {precision:.4f}")
    print(f"Recall   (teste):       {recall:.4f}")
    print(f"ROC-AUC  (CV treino):   {cv_auc_mean:.4f} ± {cv_auc_std:.4f}")
    print(f"ROC-AUC  (CV vs teste): {train_auc_mean:.4f} vs {auc:.4f}")
    print(f"Gap train/test: {abs(train_auc_mean - auc):.4f}", end=" ")
    print("Overfitting?" if abs(train_auc_mean - auc) > 0.05 else "Estável")

    return {
        "modelo": nome,
        "auc_teste": auc,
        "ks_teste": ks,
        "f1_teste": f1,
        "precision_teste": precision,
        "recall_teste": recall,
        "cv_auc_mean": cv_auc_mean,
        "cv_auc_std": cv_auc_std,
        "train_auc_mean": train_auc_mean
    }

In [ ]:
# ══════════════════════════════════════════════════════════════
# MODELO 1 — Regressão Logística
# Motivação: modelo linear interpretável, baseline inteligente
# Vantagem: coeficientes explicáveis para o comitê de crédito
# Limitação: assume relação linear entre features e target
# Requer: normalização das features (StandardScaler)
# ══════════════════════════════════════════════════════════════

lr_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        class_weight="balanced",  # trata desbalanceamento
        max_iter=1000,
        random_state=SEED,
        C=0.1                     # regularização L2 — controla multicolinearidade
    ))
])

lr_pipeline.fit(X_train, y_train)
resultado_lr = avaliar_modelo(
    "Regressão Logística", lr_pipeline,
    X_train, X_test, y_train, y_test
)

# ══════════════════════════════════════════════════════════════
# MODELO 2 — Random Forest
# Motivação: ensemble de árvores, captura não-linearidades
# Vantagem: robusto a outliers e multicolinearidade
# Limitação: menos interpretável, mais lento para treinar
# Não requer: normalização
# ══════════════════════════════════════════════════════════════

rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,              # limitar profundidade — controla overfitting
    min_samples_leaf=50,      # mínimo de amostras por folha — regularização
    class_weight="balanced",
    random_state=SEED,
    n_jobs=-1                 # usar todos os cores disponíveis
)

rf_model.fit(X_train, y_train)
resultado_rf = avaliar_modelo(
    "Random Forest", rf_model,
    X_train, X_test, y_train, y_test
)

# ══════════════════════════════════════════════════════════════
# MODELO 3 — XGBoost
# Motivação: gradient boosting — estado da arte em dados tabulares
# Vantagem: performance superior, lida bem com desbalanceamento
# Limitação: mais hiperparâmetros, risco maior de overfitting
# ══════════════════════════════════════════════════════════════

# scale_pos_weight trata desbalanceamento no XGBoost
# fórmula: n_negativos / n_positivos
scale_pw = (y_train == 0).sum() / (y_train == 1).sum()
print(f"scale_pos_weight: {scale_pw:.2f}")

xgb_model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=4,              # árvores rasas — controla overfitting
    learning_rate=0.05,       # passo pequeno — mais robusto
    subsample=0.8,            # amostragem de linhas por árvore
    colsample_bytree=0.8,     # amostragem de features por árvore
    scale_pos_weight=scale_pw,
    random_state=SEED,
    eval_metric="auc",
    n_jobs=-1
)

xgb_model.fit(X_train, y_train)
resultado_xgb = avaliar_modelo(
    "XGBoost", xgb_model,
    X_train, X_test, y_train, y_test
)

In [ ]:
# MODEL COMPARISON
resultados = [resultado_lr, resultado_rf, resultado_xgb]
df_resultados = pd.DataFrame(resultados).set_index("modelo")

print("--- TABELA COMPARATIVA DE MODELOS ---")
print(df_resultados[[
    "auc_teste", "ks_teste", "f1_teste",
    "precision_teste", "recall_teste",
    "cv_auc_mean", "cv_auc_std", "train_auc_mean"
]].round(4))

df_resultados.to_csv("../reports/model_comparison.csv")

In [ ]:
# COMPARISON OF ROC CURVES
fig, ax = plt.subplots(figsize=(10,7))

modelos_plot = [
    ("Regressao Logística", lr_pipeline, "#3498db"),
    ("Random Forest", rf_model, "#2ecc71"),
    ("XGBoost", xgb_model, "#e74c3c")
]

for nome, modelo, cor in modelos_plot:
    y_proba = modelo.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)
    ax.plot(fpr, tpr, label=f"{nome} (AUC={auc:.4f})", color=cor, linewidth=2)
    
ax.plot([0,1], [0,1], "k--", linewidth=1, label="Baseline")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curves - Model Comparison", fontweight="bold")
ax.legend(loc="lower right")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("../reports/roc_curves.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# CONCLUSION 
# XGBoost was chosen due to its combination of higher AUC and lower variance in cross-validation.
# The difference in KS compared to Random Forest is 0.002 — marginal.
# For an environment requiring regulatory explainability, we will apply SHAP values ​​in the evaluation phase.

In [ ]:
import mlflow
import mlflow.sklearn
import mlflow.xgboost
import pickle
import json

In [ ]:
from mlflow.models.signature import infer_signature
signature = infer_signature(X_train, xgb_model.predict_proba(X_train)[:,1])

mlflow.set_experiment("credit-scoring")

with mlflow.start_run(run_name="XGBoost_final_v2"):
    
    # logging params
    mlflow.log_params({
        "n_estimators": 300,
        "max_depth": 4,
        "learning_rate": 0.05,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "scale_pos_weight": round(scale_pw, 2),
        "seed" : SEED
    })
    
    # logging metrics
    mlflow.log_metrics({
        "auc_teste": resultado_xgb["auc_teste"],
        "ks_teste": resultado_xgb["ks_teste"],
        "f1_teste": resultado_xgb["f1_teste"],
        "precision_teste": resultado_xgb["precision_teste"],
        "recall_teste": resultado_xgb["recall_teste"],
        "cv_auc_mean": resultado_xgb["cv_auc_mean"],
        "cv_auc_std": resultado_xgb["cv_auc_std"]
    })
    
    # registering model
    mlflow.xgboost.log_model(xgb_model, name = "model", signature=signature, input_example=X_train.iloc[:5])
    print("Modelo registrado no MLflow com sucesso!")
    
# SAVING THE MODEL LOCALLY
with open("../models/xgboost_credit.pkl", "wb") as f:
    pickle.dump(xgb_model, f)
    
# SAVING FEATURE LIST
features_modelo_lista = X_train.columns.tolist()
with open("../models/features.json", "w") as f:
    json.dump(features_modelo_lista, f, indent=2)
    
print(f"\nModelo salvo em modelos/xgboost_credit.pkl")
print(f"Lista de features salva em modelos/features.json")
print(f"\nFeatures do Modelo ({len(features_modelo_lista)}):")
for f in features_modelo_lista:
    print(f"- {f}")